# Notebook 03 — Evaluate All Experiments
**Member: C** | Set `model.checkpoint_path` in `configs/eval.yaml` before each run.

| Experiment | `checkpoint_path` | Benchmark |
|---|---|---|
| Exp 0 zero-shot    | `null`                             | GSM8K |
| Exp 1 few-shot CoT | `null`                             | GSM8K |
| Exp 2 adaptation   | `results/checkpoints/adaptation`   | GSM8K |
| Exp 3 distillation | `results/checkpoints/distillation` | GSM8K |
| Exp 4 filtered     | `results/checkpoints/filtered`     | GSM8K |
| Exp 5 cross-task   | `results/checkpoints/filtered`     | ARC |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys
token = 'ghp_xxxxxxxxxxxx'      # paste your GitHub token each session
hf_token = 'hf_xxxxxxxxxxxx'   # paste your HuggingFace token each session

os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token  # fallback for older transformers

repo_url = f'https://{token}@github.com/TIFFMUN/SC4001-Group133.git'

if not os.path.exists('/content/SC4001-Group133/.git'):
    os.system(f'git clone {repo_url} /content/SC4001-Group133')

os.chdir('/content/SC4001-Group133')
os.system('git fetch origin main')
os.system('git reset --hard origin/main')

import importlib
for mod in list(sys.modules.keys()):
    if mod.startswith('src'):
        del sys.modules[mod]
importlib.invalidate_caches()

!pip install -r requirements.txt -q
!pip install -e . -q

In [ ]:
# Redirect HF cache + data/ + results/ to Drive — run before any HF imports
import importlib
importlib.invalidate_caches()
from src.setup_cache import setup
setup()

In [ ]:
# Exp 0 — zero-shot baseline
MAX_SAMPLES = 5  # set to 5 for quick test, 100 for full eval, None for all 1319

from src.baselines.cot_prompting import run_baseline
from src.distillation.filter_traces import extract_answer, answers_match
from src.evaluation.metrics import compute_summary_stats, save_results

results = run_baseline(mode='zero_shot', dataset_name='gsm8k', max_samples=MAX_SAMPLES)

for r in results:
    r['is_correct'] = answers_match(extract_answer(r['prediction']), r['ground_truth'])

save_results(results, 'results/exp0_zero_shot_gsm8k.csv')
print(compute_summary_stats(results))

In [ ]:
# Exp 1 + prompt ablation — all variants including few-shot CoT baseline
MAX_SAMPLES = 5  # set to 5 for quick test, 100 for full eval, None for all 1319

import json
from pathlib import Path
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from tqdm import tqdm

from src.baselines.cot_prompting import (
    build_few_shot_prompt,
    build_one_shot_prompt,
    build_zero_shot_cot_prompt,
    build_concise_prompt,
    build_format_reinforced_prompt,
)
from src.distillation.filter_traces import extract_answer, answers_match
from src.evaluation.metrics import compute_latency, compute_summary_stats, save_results, print_summary

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
model.eval()

test_data = load_dataset("openai/gsm8k", "main")["test"].select(range(MAX_SAMPLES))

prompt_variants = {
    "exp1_few_shot":            build_few_shot_prompt,
    "exp1b_one_shot":           build_one_shot_prompt,
    "exp1c_zs_cot":             build_zero_shot_cot_prompt,
    "exp1d_concise":            build_concise_prompt,
    "exp1e_format_reinforced":  build_format_reinforced_prompt,
}

all_stats = {}
for name, prompt_fn in prompt_variants.items():
    results = []
    with torch.no_grad():
        for sample in tqdm(test_data, desc=name):
            prompt = prompt_fn(sample["question"], tokenizer)
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            input_len = inputs["input_ids"].shape[1]
            latency_ms, output_ids = compute_latency(
                model=model, inputs=inputs, max_new_tokens=256, do_sample=False
            )
            generated_ids = output_ids[0][input_len:]
            prediction = tokenizer.decode(generated_ids, skip_special_tokens=True)
            predicted_answer = extract_answer(prediction)
            is_correct = answers_match(predicted_answer, sample["answer"])
            results.append({
                "question": sample["question"],
                "ground_truth": sample["answer"],
                "prediction": prediction,
                "predicted_answer": predicted_answer,
                "is_correct": is_correct,
                "latency_ms": latency_ms,
                "token_count": len(generated_ids),
            })
    stats = compute_summary_stats(results)
    print_summary(name, stats)
    save_results(results, f"results/{name}_gsm8k.csv")
    stats_path = Path("results/summary_results") / f"{name}_gsm8k_summary.json"
    stats_path.parent.mkdir(parents=True, exist_ok=True)
    with open(stats_path, "w") as f:
        json.dump({"experiment": name, **stats}, f, indent=2)
    all_stats[name] = stats

print("\n=== COMPARISON ===")
print(f"{'Variant':<30} {'Accuracy':>10} {'Mean Tokens':>12}")
print("-" * 55)
print(f"{'exp0_zero_shot (baseline)':<30} {'29.00%':>10} {'241.8':>12}")
for name, s in all_stats.items():
    print(f"{name:<30} {s['accuracy']:>10.2%} {s['mean_token_count']:>12.1f}")

In [ ]:
# Exp 2-4 — set checkpoint_path in configs/eval.yaml, update EXPERIMENT_NAME, then run
EXPERIMENT_NAME = 'exp2_adaptation_r16'  # change per run
MAX_SAMPLES = 5  # set to 5 for quick test, 100 for full eval, None for all 1319

from src.evaluation.eval_gsm8k import evaluate_gsm8k
evaluate_gsm8k(experiment_name=EXPERIMENT_NAME, max_samples=MAX_SAMPLES)

In [ ]:
# Exp 5 — ARC cross-task transfer (uses Exp 4 checkpoint)
from src.evaluation.eval_arc import evaluate_arc
evaluate_arc(experiment_name='exp5_d3_cross_task')

In [ ]:
# List all saved CSVs on Drive
csvs = [f for f in os.listdir('/content/drive/MyDrive/SC4001/results') if f.endswith('.csv')]
for f in sorted(csvs):
    print(f)

## Exp 6 — Inference-gated budget router (optional, run after Exps 0–5)

Prerequisite: set `router.enabled: true` and `router.checkpoint_path` in `configs/eval.yaml`.

The router sweeps a range of complexity thresholds and saves a Pareto-ready CSV to
`results/exp6_router_sweep.csv`. Use `max_samples=50` for a quick sanity check first.

In [ ]:
# Exp 6 — budget router threshold sweep
# Step 1: quick sanity check with 50 samples
from src.evaluation.eval_router import run_router_sweep

sweep_df = run_router_sweep(experiment_name='exp6_router', max_samples=50)
print(sweep_df[['threshold', 'accuracy', 'mean_token_count', 'pct_easy']].to_string(index=False))

In [ ]:
# Exp 6 — full sweep (set max_samples: null in eval.yaml for full 1319-sample eval)
# After confirming the sanity check above looks reasonable, run:
sweep_df_full = run_router_sweep(experiment_name='exp6_router_full', max_samples=None)
print(sweep_df_full[['threshold', 'accuracy', 'mean_token_count', 'pct_easy']].to_string(index=False))